In [13]:
import numpy as np
import pandas as pd

df = pd.read_csv('/content/train_numeric.csv', nrows=100000)

measurement_cols = [col for col in df.columns if col not in ['Id', 'Response']]

variances = np.nanvar(df[measurement_cols].to_numpy(), axis=0)
constant_cols = [measurement_cols[i] for i, var in enumerate(variances) if var == 0 or np.isnan(var)]

print("Measurement count:", len(measurement_cols))
print("Constant measurement count:", len(constant_cols))

Measurement count: 968
Constant measurement count: 36


In [14]:
station_groups = {}

for col in measurement_cols:
    parts = col.split('_')
    station_key = parts[0] + '_' + parts[1]

    if station_key not in station_groups:
        station_groups[station_key] = []

    station_groups[station_key].append(col)

for station, cols in sorted(station_groups.items()):
    print(f"{station}: {len(cols)}")

L0_S0: 12
L0_S1: 2
L0_S10: 12
L0_S11: 12
L0_S12: 12
L0_S13: 2
L0_S14: 9
L0_S15: 9
L0_S16: 2
L0_S17: 2
L0_S18: 3
L0_S19: 3
L0_S2: 9
L0_S20: 3
L0_S21: 14
L0_S22: 14
L0_S23: 14
L0_S3: 9
L0_S4: 2
L0_S5: 2
L0_S6: 3
L0_S7: 3
L0_S8: 3
L0_S9: 12
L1_S24: 229
L1_S25: 284
L2_S26: 14
L2_S27: 14
L2_S28: 14
L3_S29: 53
L3_S30: 68
L3_S31: 4
L3_S32: 1
L3_S33: 10
L3_S34: 4
L3_S35: 8
L3_S36: 8
L3_S37: 4
L3_S38: 3
L3_S39: 4
L3_S40: 8
L3_S41: 14
L3_S43: 8
L3_S44: 8
L3_S45: 5
L3_S47: 11
L3_S48: 6
L3_S49: 7
L3_S50: 7
L3_S51: 4


In [15]:
selected_station = 'L1_S25'
selected_cols = station_groups[selected_station]

valid_counts = df[selected_cols].count()

counts_df = pd.DataFrame({
    'Measurement Column': valid_counts.index,
    'Valid Sample Count (Non-Null)': valid_counts.values
}).sort_values(by='Valid Sample Count (Non-Null)', ascending=False, kind='stable')

print(f"Selected Station: {selected_station}")
print(f"Total Measurement Columns: {len(selected_cols)}")
print("\n--- TOP 10 ---")
display(counts_df.head(10))

print("\n--- LAST 10 ---")
display(counts_df.tail(10))

Selected Station: L1_S25
Total Measurement Columns: 284

--- TOP 10 ---


,Measurement Column,Valid Sample Count (Non-Null)
0,L1_S25_F1855,231
1,L1_S25_F1858,231
6,L1_S25_F1881,231
7,L1_S25_F1885,231
8,L1_S25_F1890,230
9,L1_S25_F1892,230
10,L1_S25_F1894,230
2,L1_S25_F1865,229
3,L1_S25_F1869,229
4,L1_S25_F1873,229



--- LAST 10 ---


,Measurement Column,Valid Sample Count (Non-Null)
202,L1_S25_F2724,10
203,L1_S25_F2726,10
68,L1_S25_F2181,4
69,L1_S25_F2184,4
70,L1_S25_F2187,4
71,L1_S25_F2190,4
72,L1_S25_F2193,4
73,L1_S25_F2196,4
74,L1_S25_F2199,4
75,L1_S25_F2202,4


In [16]:
top_15_cols = counts_df.head(15)['Measurement Column'].tolist()

A_df = df[top_15_cols].dropna()
A = A_df.to_numpy()

print("Selected measurements:")
for col in top_15_cols:
    print(col)

print("\nMatrix shape:", A.shape)

Selected measurements:
L1_S25_F1855
L1_S25_F1858
L1_S25_F1881
L1_S25_F1885
L1_S25_F1890
L1_S25_F1892
L1_S25_F1894
L1_S25_F1865
L1_S25_F1869
L1_S25_F1873
L1_S25_F1877
L1_S25_F2136
L1_S25_F2144
L1_S25_F2147
L1_S25_F2152

Matrix shape: (227, 15)


In [17]:
print(A.shape)
print(f"Rank: {np.linalg.matrix_rank(A)}")

(227, 15)
Rank: 15


In [18]:
means = np.mean(A, axis=0)
A_centered = A - means

new_means = np.mean(A_centered, axis=0)
print("Means after centering:")
print(new_means)

A_unit = A_centered / np.linalg.norm(A_centered, axis=0)

print("\nColumn norms after normalization:")
print(np.linalg.norm(A_unit, axis=0))

Means after centering:
[ 0.00000000e+00 -5.37993536e-18 -9.78170066e-19  9.17034437e-19
  1.34498384e-18  6.35810543e-18  1.34498384e-18 -1.95634013e-17
 -1.95634013e-18 -5.73146523e-21  4.12665496e-19  1.95634013e-18
  1.22271258e-18 -6.84719046e-18 -3.91268026e-18]

Column norms after normalization:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [19]:
U, s, Vh = np.linalg.svd(A_unit, full_matrices=False)

print("Singular Values:")
print(s)

relative_s = s / s[0]
print("\nRelative Singular Values:")
print(relative_s)

Singular Values:
[2.22459484 1.73774589 1.18993935 1.11533581 1.05425563 0.99022792
 0.94907161 0.78873278 0.67462047 0.38488473 0.36474728 0.12806747
 0.06268504 0.00433883 0.00237589]

Relative Singular Values:
[1.         0.78115163 0.53490161 0.50136582 0.47390905 0.44512731
 0.42662673 0.35455121 0.30325543 0.17301341 0.16396122 0.05756889
 0.02817818 0.00195039 0.00106801]


In [20]:
s_squared = s**2
total_s_squared = np.sum(s_squared)
cumulative_s_squared = np.cumsum(s_squared)

cumulative_ratio = cumulative_s_squared / total_s_squared
print("Cumulative squared singular-value ratio:")
print(cumulative_ratio)

threshold = 0.95
num_components_95_percent = np.argmax(cumulative_ratio >= threshold) + 1

print("\nDirections needed for 95%:", num_components_95_percent)

Cumulative squared singular-value ratio:
[0.32992148 0.53123887 0.62563591 0.70856751 0.7826645  0.84803459
 0.90808372 0.94955701 0.97989787 0.98977362 0.99864299 0.99973641
 0.99999837 0.99999962 1.        ]

Directions needed for 95%: 9


In [21]:
reconstruction_residuals = {}

for i in range(A_unit.shape[1]):

    x = A_unit[:, i]
    B = np.delete(A_unit, i, axis=1)

    beta = np.linalg.lstsq(B, x, rcond=None)[0]
    x_reconstructed = B @ beta

    residual = np.linalg.norm(x - x_reconstructed)

    measurement_name = top_15_cols[i]
    reconstruction_residuals[measurement_name] = residual

sorted_residuals = sorted(reconstruction_residuals.items(), key=lambda item: item[1])

print("Reconstruction Residuals (L2 Norm):\n")

for measurement, residual in sorted_residuals:
    print(f"{measurement}: {residual:.6f}")

strongest_candidate_name = sorted_residuals[0][0]
lowest_residual = sorted_residuals[0][1]

print("\nStrongest redundancy candidate:")
print(strongest_candidate_name)
print("Residual:", lowest_residual)

Reconstruction Residuals (L2 Norm):

L1_S25_F1869: 0.003367
L1_S25_F2147: 0.003369
L1_S25_F2152: 0.006082
L1_S25_F1885: 0.006089
L1_S25_F1890: 0.079209
L1_S25_F1892: 0.092615
L1_S25_F1894: 0.146744
L1_S25_F1855: 0.440597
L1_S25_F1877: 0.506855
L1_S25_F1873: 0.834734
L1_S25_F1865: 0.872432
L1_S25_F1858: 0.917330
L1_S25_F2144: 0.970164
L1_S25_F2136: 0.974591
L1_S25_F1881: 0.980028

Strongest redundancy candidate:
L1_S25_F1869
Residual: 0.003367327757663094


In [22]:
strongest_candidate_index = top_15_cols.index(strongest_candidate_name)
target_vector = A_unit[:, strongest_candidate_index]

similarities = {}

for i, measurement_name in enumerate(top_15_cols):

    if measurement_name != strongest_candidate_name:

        other_vector = A_unit[:, i]
        similarity = np.dot(target_vector, other_vector)

        similarities[measurement_name] = similarity

sorted_similarities = sorted(
    similarities.items(),
    key=lambda item: abs(item[1]),
    reverse=True
)

print("Dot Product Similarities:\n")

for measurement, similarity in sorted_similarities:
    print(f"{measurement}: {similarity:.6f}")

Dot Product Similarities:

L1_S25_F2147: 0.999994
L1_S25_F1855: 0.584910
L1_S25_F1877: 0.489203
L1_S25_F1873: -0.248414
L1_S25_F2144: -0.044550
L1_S25_F1894: -0.037637
L1_S25_F2152: 0.035123
L1_S25_F1885: 0.034750
L1_S25_F1890: -0.034744
L1_S25_F1892: -0.027308
L1_S25_F1858: -0.025002
L1_S25_F1881: 0.023850
L1_S25_F2136: 0.019762
L1_S25_F1865: 0.012827


In [23]:
top_4_similar_names = [name for name, similarity in sorted_similarities[:4]]

small_group_names = [strongest_candidate_name] + top_4_similar_names
small_group_indices = [top_15_cols.index(name) for name in small_group_names]

A_small_group = A_unit[:, small_group_indices]

Gram_matrix = A_small_group.T @ A_small_group

Gram_matrix_df = pd.DataFrame(
    Gram_matrix,
    index=small_group_names,
    columns=small_group_names
)

display(Gram_matrix_df)

,L1_S25_F1869,L1_S25_F2147,L1_S25_F1855,L1_S25_F1877,L1_S25_F1873
L1_S25_F1869,1.000000,0.999994,0.584910,0.489203,-0.248414
L1_S25_F2147,0.999994,1.000000,0.584590,0.488904,-0.248181
L1_S25_F1855,0.584910,0.584590,1.000000,0.848510,0.194765
L1_S25_F1877,0.489203,0.488904,0.848510,1.000000,0.179283
L1_S25_F1873,-0.248414,-0.248181,0.194765,0.179283,1.000000


In [24]:
s_small_group = np.linalg.svd(A_small_group, compute_uv=False)
relative_s_small_group = s_small_group / s_small_group[0]

print("Singular Values of Small Group:")
print(s_small_group)

print("\nRelative Singular Values of Small Group:")
print(relative_s_small_group)

Singular Values of Small Group:
[1.73363803 1.16710462 0.70232272 0.37296496 0.0024351 ]

Relative Singular Values of Small Group:
[1.         0.67321125 0.40511497 0.21513427 0.00140462]
